<a href="https://colab.research.google.com/github/amandaliram/pln_nlp_repository/blob/main/A10_RP07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Roteiro de Práticas 07:** Busca Semântica e Similaridade de Cosseno

A transição da simples contagem de frequência de termos (TF-IDF) para a análise de proximidade vetorial representa o verdadeiro divisor de águas entre os sistemas legados de busca por palavra-chave e os modernos motores de busca por intenção. Enquanto o TF-IDF nos permite identificar a relevância de um termo através da atenuação logarítmica (log 10), a Similaridade de Cosseno permite que uma aplicação compreenda a orientação semântica de uma frase. Ao tratar textos como coordenadas espaciais, superamos a barreira da correspondência exata de caracteres para alcançar a recuperação de informação baseada no significado latente.

Nesta etapa, deixamos de apenas contar palavras para interpretar a orientação de perfis matemáticos, transformando o "corpus higienizado em vetores capazes de responder a intenções de busca complexas no contexto utilizado. Ao final desta prática, você deverá ser capaz de:

1. **Implementar** o cálculo da Similaridade de Cosseno utilizando álgebra linear para medir a proximidade entre documentos.
2. **Diferenciar** métricas de distância angular (Cosseno) de métricas de magnitude (Euclidiana), justificando a escolha da primeira para coleções de texto.
3. **Construir** um motor de busca funcional que opere sobre matrizes esparsas ponderadas por TF-IDF.

### **1. Fundamentação: A matemática da Busca**

Em domínios textuais, como avaliação de um e-commerce, a variação de magnitude (tamanho) dos documentos é um desafio constante. Um review pode conter três palavras ("Bateria muito boa"), enquanto outro pode ser um relato de dois parágrafos. Se utilizássemos a distância euclidiana pura, o review longo seria considerado "distante" do curto apenas pelo volume de palavras, mesmo que ambos tratassem do mesmo tema. A Similaridade de Cosseno resolve essa disparidade ao focar na **direção** do vetor, sendo invariante ao comprimento do documento.

**1.1 Conceito de Espaço Vetorial**

A "Faxina Digital" (Tokenização e Lematização) reduz a dimensionalidade do problema, eliminando ruídos. O vocabulário limpo torna-se a base de um mapa multidimensional. Cada documento é um vetor (ou um ponto) neste espaço, onde as coordenadas são definidas pelos pesos TF-IDF calculados com a escala logarítmica (log 10).

**1.2 A Fórmula do Cosseno**

A similaridade é o cosseno do ângulo entre dois vetores (A e B). O resultado é normalizado entre 0 (vetores ortogonais, sem similaridade) e 1 (vetores colineares, máxima similaridade).

$$ \text{similarity} = \cos(\theta) = \frac{A \cdot B}{\|A\| \|B\|} $$

Aqui, $\|A\|$ e $\|B\|$ representam a **Norma L2 (Norma Euclidiana)** dos vetores. Em engenharia de PLN, a biblioteca `scikit-learn` otimiza este cálculo utilizando estruturas **CSR (Compressed Sparse Row)**, o que é vital para economizar RAM e custos de processamento em nuvem ao lidar com milhares de textos como os reviews.

Imagine que dois exploradores partem da origem em um campo aberto. Um caminha 100 metros (um texto curto) e outro caminha 10 km (um texto longo). Se ambos caminham exatamente para o "Norte", a distância em linha reta entre eles é enorme, mas a **direção** é idêntica. Em busca semântica, a direção representa o assunto (ex: "bateria"). A similaridade de cosseno ignora o quão longe cada um foi e foca no ângulo: se o ângulo é zero, o assunto é o mesmo.

**1.3 O que é a Busca Semântica?**

Enquanto a busca tradicional em texto completo (*full-text search*) tenta encontrar os documentos que usam as **palavras exatas** da sua consulta, a busca semântica leva em consideração o **significado** das palavras tanto na sua consulta quanto nos documentos pesquisados. Ela utiliza as representações vetoriais densas (*Embeddings*): as palavras com significados semelhantes estarão próximas no espaço vetorial. Assim, o sistema retorna resultados extremamente precisos, mesmo que você não saiba exatamente quais palavras o autor do texto utilizou.

A ideia por trás da busca semântica é encontrar os documentos que tenham a maior similaridade semântica com a sua consulta. Transformamos a frase que o usuário digitou na barra de pesquisa em um vetor numérico (usando modelos como Word2Vec, ou Embeddings Contextuais poderosos como o BERT) e comparamos esse vetor de consulta com os vetores dos documentos da base de dados, tipicamente calculando a **Similaridade de Cosseno** entre eles.

### **2. Procedimentos Experimentais (Mão na Massa)**

**2.1 Busca Tradicional**  
  

2.1.1 **Exemplo 1:** Busca utilizando List Comprehension

In [1]:
# Nosso "corpus" (pequeno banco de dados de documentos)
documentos = [
    "A linguagem Python é muito fácil de aprender.",
    "O leão é considerado o rei da selva.",
    "Gatos são felinos muito ágeis e independentes.",
    "A busca semântica entende o significado real do texto."
]

In [2]:
# Nossa consulta (palavra-chave)
consulta = "felino"

# Busca Tradicional (Correspondência Exata) usando List Comprehension
# Convertemos a consulta e o texto do documento para minúsculo [2]
resultados = [doc for doc in documentos if consulta.lower() in doc.lower()]

In [3]:
print(f"Resultados da busca por: '{consulta}'")
for res in resultados:
    print("-", res)

Resultados da busca por: 'felino'
- Gatos são felinos muito ágeis e independentes.


**Obs.:** a busca tradicional em texto completo procura exatamente a string (ou sequência de caracteres) que você digitou dentro do texto dos documentos pesquisados

2.1.2 **Exemplo 2:** Busca utilizando mapeamento em conjunto com expressões regulares

In [4]:
import re

In [5]:
# Nosso "corpus" bruto, simulando textos reais e despadronizados
documentos = [
    "O C.E.O. da empresa publicou novas diretrizes para o sistema.",
    "O candidato informou que é SELF EMPLOYED em sua inscrição.",
    "João e maria foram aprovados no concurso público.",
    "A taxa de juros do mercado financeiro aumentou drasticamente."
]

In [6]:
# 1. TÉCNICA DE MAPEAMENTO (Dicionário de Padronização)
# Mapeando como o usuário pode ter escrito (chave) para o padrão do sistema (valor)
dicionario_mapeamento = {
    "C.E.O.": "CEO",
    "SELF EMPLOYED": "SELF-EMPLOYED"
}

# Aplicando a limpeza (mapeamento) antes da busca
documentos_normalizados = []
for doc in documentos:
    # O método replace substitui as ocorrências de uma string por outra [8]
    for termo_original, termo_padronizado in dicionario_mapeamento.items():
        doc = doc.replace(termo_original, termo_padronizado)
    documentos_normalizados.append(doc)

# Neste ponto, "C.E.O." virou "CEO" e "SELF EMPLOYED" virou "SELF-EMPLOYED".

In [7]:
# 2. INJEÇÃO DE EXPRESSÃO REGULAR (Busca Otimizada)
# Vamos buscar por cargos ou nomes usando o metacaractere '|' para representar "OU"
padrao_busca = r'CEO|SELF-EMPLOYED|João|Maria'

print("--- Resultados da Busca ---")

for doc in documentos_normalizados:
    # re.findall retorna todas as ocorrências encontradas na string em forma de lista [9]
    # flags=re.I garante que "maria" (minúsculo) dê match com "Maria" do nosso padrão
    matches = re.findall(padrao_busca, doc, flags=re.I)

    # Se a lista de matches não estiver vazia, significa que a busca tradicional obteve sucesso
    if matches:
        print(f"Encontramos o(s) termo(s) {matches} no texto: -> '{doc}'")

--- Resultados da Busca ---
Encontramos o(s) termo(s) ['CEO'] no texto: -> 'O CEO da empresa publicou novas diretrizes para o sistema.'
Encontramos o(s) termo(s) ['SELF-EMPLOYED'] no texto: -> 'O candidato informou que é SELF-EMPLOYED em sua inscrição.'
Encontramos o(s) termo(s) ['João', 'maria'] no texto: -> 'João e maria foram aprovados no concurso público.'


**Obs:** A construção deste script segue a seguinte estrutura: Mapeamento (um dicionário para a limpeza de termos problemáticos); Busca com Regex, Metacaracteres, Ignorando a Caixa (Case).

**2.2 Busca Semântica**  

2.2.1 **Exemplo 3:** A Matemática da Busca Semântica

In [8]:
import numpy as np

In [9]:
# Representação de frequências (Ex: [bateria, tela, preco])
doc_a = np.array([3, 1, 0])
doc_b = np.array([2, 0, 1])

In [10]:
def calcular_cosseno(v1, v2):
    # Produto Escalar: A . B
    dot_product = np.dot(v1, v2)

    # Magnitudes (Norma L2 / Norma Euclidiana)
    norm_a = np.linalg.norm(v1)
    norm_b = np.linalg.norm(v2)

    # Similaridade: Invariante à magnitude, focada na orientação
    return dot_product / (norm_a * norm_b)

In [11]:
resultado = calcular_cosseno(doc_a, doc_b)

print(f"Similaridade de Cosseno (Álgebra Pura): {resultado:.4f}")

Similaridade de Cosseno (Álgebra Pura): 0.8485


2.2.2 **Exemplo 4:** Busca Semântica com similaridade cosseno

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
# Corpus de reviews tecnológicos
reviews = [
    "bateria excelente tela excelente",
    "bateria ruim e descarrega rapido",
    "tela quebrou facil",
    "celular bom mas a entrega atrasou"
]

In [14]:
# Vetorização: Scikit-learn usa Matrizes Esparsas (CSR) por padrão
vectorizer = TfidfVectorizer()
matrix_esparsa = vectorizer.fit_transform(reviews)

In [15]:
# Processando a Query (consulta)
query = ["problema bateria"]
query_vector = vectorizer.transform(query)

In [16]:
# O Scikit-learn é otimizado para similaridade em matrizes esparsas
scores = cosine_similarity(query_vector, matrix_esparsa)
best_match_idx = scores.argmax()

In [17]:
print(f"--- INSIGHT SOCIAL LISTENING ---")
print(f"Review Identificado: {reviews[best_match_idx]}")
print(f"Confiança Semântica: {scores[0][best_match_idx]:.4f}")

print("\nRanking de Relevância:")
for i in scores.argsort()[0][::-1]:
    print(f"Score: {scores[0][i]:.4f} | Review: {reviews[i]}")

--- INSIGHT SOCIAL LISTENING ---
Review Identificado: bateria ruim e descarrega rapido
Confiança Semântica: 0.4143

Ranking de Relevância:
Score: 0.4143 | Review: bateria ruim e descarrega rapido
Score: 0.3443 | Review: bateria excelente tela excelente
Score: 0.0000 | Review: tela quebrou facil
Score: 0.0000 | Review: celular bom mas a entrega atrasou


2.2.3 **Exemplo 5:** Busca Semântica com modelos pré treinados

In [18]:
!pip install spacy

In [19]:
# Baixamos o modelo de linguagem pré-treinado em português
# Este comando é necessário para baixar o modelo se ele ainda não estiver presente.
!python -m spacy download pt_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 21.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [20]:
import spacy

# 1. Carregamos um modelo de linguagem pré-treinado em português
nlp = spacy.load("pt_core_news_md")

In [21]:
# Nosso "corpus" (banco de dados)
documentos = [
    "A linguagem Python é muito fácil de aprender.",
    "O leão é considerado o rei da selva.",
    "Gatos são animais muito ágeis e independentes.",
    "A busca semântica entende o significado real do texto."
]

In [22]:
# 2. Representação da Consulta:
# Transformamos a palavra "felino" em um vetor matemático denso
consulta = nlp("felino")

In [23]:
print("--- Resultados da Busca Semântica ---")

for texto in documentos:
    # 3. Representação do Documento:
    # O spaCy calcula o vetor médio que representa toda a frase
    doc_vetorizado = nlp(texto)

    # 4. Cálculo da Similaridade:
    # Calcula a Similaridade de Cosseno entre o vetor da consulta e o da frase
    similaridade = consulta.similarity(doc_vetorizado)

    print(f"Score de Similaridade: {similaridade:.2f} -> '{texto}'")

--- Resultados da Busca Semântica ---
Score de Similaridade: 0.06 -> 'A linguagem Python é muito fácil de aprender.'
Score de Similaridade: 0.19 -> 'O leão é considerado o rei da selva.'
Score de Similaridade: 0.12 -> 'Gatos são animais muito ágeis e independentes.'
Score de Similaridade: -0.01 -> 'A busca semântica entende o significado real do texto.'


## **3. Desafio**

Você deve criar um script de teste que compare o desempenho de duas abordagens de busca para uma consulta "traiçoeira".

1. **A Query do Usuário:** "O aparelho descarrega muito rápido".
2. **O Corpus de Teste:** Garanta que seu banco de dados contenha um review que diga: "A autonomia da bateria é decepcionante".

### Tarefas

* **Etapa 1 (Busca por Frequência):** Execute a busca utilizando sua matriz **TF-IDF** atual. Documente o escore de similaridade. (Dica: Por que o resultado provavelmente será 0 ou muito baixo?)
* **Etapa 2 (Busca por Significado):** Utilize o modelo pré-treinado (como o pt_core_news_md do spaCy) para gerar **Word Embeddings** (vetores densos) tanto para a query quanto para o review.
* **Etapa 3 (Análise Comparativa):** Calcule a Similaridade de Cosseno entre os vetores densos e compare com o resultado da Etapa 1.

### O que deve ser entregue (Insights de Especialista):

Responda, em formato de comentário no código ou relatório técnico:

1. Ortogonalidade: Na Etapa 1, os vetores eram ortogonais? Justifique com base na ausência de termos compartilhados.
2. Hipótese Distribucional: Como os *Embeddings* conseguiram "saber" que "descarrega rápido" e "autonomia decepcionante" habitam o mesmo "bairro geográfico" no mapa das palavras?.
3. Sugestão de Escalabilidade: Se você tivesse 1 milhão de reviews, como resolveria o custo computacional de comparar a query com cada um deles? Pesquise sobre técnicas ANN (Approximate Nearest Neighbors).

In [24]:
# ==============================================================================
# DESAFIO FINAL: TF-IDF vs Word Embeddings (A Busca Traiçoeira)
# ==============================================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import spacy

# Configurando o cenário do Desafio
query_usuario = "O aparelho descarrega muito rápido"
corpus_teste = [
    "A tela do celular quebrou na primeira queda.",
    "A autonomia da bateria é decepcionante.", # O review alvo
    "O design é bonito, mas o preço é muito alto."
]

print(f'Query do Usuário: "{query_usuario}"\n')

# ---------------------------------------------------------
# ETAPA 1: Busca por Frequência (TF-IDF)
# ---------------------------------------------------------
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus_teste)
query_tfidf = vectorizer.transform([query_usuario])

# Calculando a similaridade de cosseno na matriz esparsa
score_tfidf = cosine_similarity(query_tfidf, tfidf_matrix)

print("--- ETAPA 1: Resultado TF-IDF ---")
# Pegando o score especificamente do índice 1 (nosso review alvo)
print(f"Score de Similaridade: {score_tfidf[0][1]:.4f}")
print("Motivo: A máquina procurou 'aparelho', 'descarrega', 'rápido'. Não achou nenhuma na frase alvo.\n")

# ---------------------------------------------------------
# ETAPAS 2 e 3: Busca por Significado (Embeddings / spaCy)
# ---------------------------------------------------------
# Carregando o modelo (certifique-se de ter rodado o download do pt_core_news_md antes)
nlp = spacy.load("pt_core_news_md")

doc_query = nlp(query_usuario)
doc_review_alvo = nlp(corpus_teste[1])

# O spaCy já possui o método .similarity() embutido que calcula o cosseno dos vetores densos
score_spacy = doc_query.similarity(doc_review_alvo)

print("--- ETAPAS 2 e 3: Resultado Word Embeddings ---")
print(f"Score de Similaridade: {score_spacy:.4f}")
print("Motivo: A máquina ignorou as letras e comparou os 'bairros semânticos'. O contexto é o mesmo!")

Query do Usuário: "O aparelho descarrega muito rápido"

--- ETAPA 1: Resultado TF-IDF ---
Score de Similaridade: 0.0000
Motivo: A máquina procurou 'aparelho', 'descarrega', 'rápido'. Não achou nenhuma na frase alvo.

--- ETAPAS 2 e 3: Resultado Word Embeddings ---
Score de Similaridade: 0.1632
Motivo: A máquina ignorou as letras e comparou os 'bairros semânticos'. O contexto é o mesmo!


## 💡 Insights de Especialista (Respostas do Desafio)

**1. Ortogonalidade:** Na Etapa 1, os vetores eram ortogonais? Justifique com base na ausência de termos compartilhados.
**Resposta:** Sim, os vetores eram perfeitamente ortogonais (formando um ângulo de 90° entre si, cujo cosseno é exatamente 0.0). O TF-IDF trabalha com correspondência exata de tokens. Como a query ("aparelho", "descarrega", "rápido") e o review alvo ("autonomia", "bateria", "decepcionante") não compartilham **nenhuma** palavra em comum, os vetores não se cruzam em nenhum eixo dimensional da matriz esparsa. O algoritmo é cego para sinônimos.

---

**2. Hipótese Distribucional:** Como os *Embeddings* conseguiram "saber" que "descarrega rápido" e "autonomia decepcionante" habitam o mesmo "bairro geográfico" no mapa das palavras?
**Resposta:** Graças ao treinamento prévio em bilhões de frases reais (como a Wikipedia e notícias). Nesses textos massivos, palavras como "descarrega" e "rápido" costumam aparecer vizinhas a palavras como "bateria", "autonomia" e "celular". Pela Hipótese Distribucional (*"uma palavra é conhecida pela companhia que mantém"*), a rede neural ajustou as coordenadas matemáticas desses termos para a mesma região do espaço multidimensional. Portanto, mesmo usando palavras totalmente diferentes, a direção semântica das duas frases aponta para o mesmo conceito de "problema de energia".

---

**3. Sugestão de Escalabilidade:** Se você tivesse 1 milhão de reviews, como resolveria o custo computacional de comparar a query com cada um deles?
**Resposta:** Comparar um vetor denso contra 1 milhão de outros vetores usando a Similaridade de Cosseno exata (força bruta) é computacionalmente inviável e caro para sistemas em tempo real. A solução de arquitetura é utilizar algoritmos de **ANN (Approximate Nearest Neighbors - Vizinhos Mais Próximos Aproximados)**.
Em vez de calcular a distância contra *todos* os documentos, técnicas como **HNSW** (Hierarchical Navigable Small World) ou bibliotecas como o **FAISS** (da Meta) organizam os vetores em grafos ou partições. Isso permite que a busca pule regiões irrelevantes do espaço e encontre os reviews mais similares em frações de milissegundos, trocando uma ínfima margem de precisão por um ganho massivo de velocidade e escalabilidade.